<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Grouped-Query-Attention/grouped-query-attention-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("grouped-query-attention", ...)`


# Implement Attention from Scratch
### 🧠 Problem Statement
Standard Multi-Head Attention (MHA) assigns a separate query, key, and value projection to each attention head. But that’s not always the most efficient approach.

Enter **Grouped Query Attention (GQA)** — a clever mechanism where you use more query heads than key-value heads. This reduces compute/memory costs while still allowing for fine-grained query specialization.

Your task is to **implement GQA from scratch** and validate it against PyTorch’s `MultiheadAttention` under the special case where GQA behaves identically to MHA (i.e., when `num_query_heads == num_query_groups`).

---

### ✅ Requirements

1. **Define the GQA Mechanism**
   - Create a function `grouped_query_attention(q, k, v, num_query_groups, d_model, mask=None)`.
   - Project `q`, `k`, and `v` using linear layers:
     - Q projection → all query heads.
     - K/V projection → shared across grouped key/value heads.
   - Use `repeat_interleave()` to expand grouped K/V heads to match the number of Q heads.

2. **Compute Attention**
   - Apply scaled dot-product attention using `Q @ Kᵀ / sqrt(d_head)`.
   - Support optional masking.
   - Return output by concatenating heads and applying the output projection.

3. **Validate Against MHA**
   - Test your implementation using synthetic tensors.
   - Compare your output to `torch.nn.MultiheadAttention` where GQA degenerates to MHA (`num_query_heads == num_query_groups`).
   - Assert that both outputs match numerically.

---

### 📏 Constraints

- ✅ Use only PyTorch (no external libraries like xformers or HuggingFace).
- ✅ Output shape must be `(batch_size, seq_len, d_model)`.
- ✅ Support optional attention masking.
- ✅ Validate output against `torch.nn.MultiheadAttention` for correctness.

---

<details>
  <summary>💡 Hint</summary>

  - Use `nn.Linear(d_model, d_model)` for projecting `q`, `k`, and `v`.
  - When `num_query_heads > num_query_groups`, use `.repeat_interleave()` to duplicate each group’s `K`/`V` to match query head count.
  - Final output: reshape the multi-head outputs to `(batch_size, seq_len, d_model)` and apply the output projection layer.
  - Test with `num_query_heads == num_query_groups` to confirm it behaves like MHA.

</details>

---

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

torch.Size([3, 4, 8])


In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def grouped_query_attention(q, k, v, num_query_heads, num_query_groups, d_model,
                            mask=None):
    """
    Implements Grouped Query Attention (GQA).

    Queries keep `num_query_heads` heads, but keys and values only have
    `num_query_groups` heads; each K/V head is shared by
    `num_query_heads // num_query_groups` query heads. With
    num_query_groups == num_query_heads this is ordinary multi-head attention,
    and with num_query_groups == 1 it is multi-query attention.

    Args:
        q, k, v (Tensor): (batch_size, seq_len, d_model)
        num_query_heads (int): number of query heads
        num_query_groups (int): number of key/value heads; must divide num_query_heads
        d_model (int): total embedding dimension
        mask (Tensor, optional): broadcastable to (batch, heads, seq, seq);
            positions equal to 0 are not attended to

    Returns:
        Tensor: (batch_size, seq_len, d_model)
    """
    assert num_query_heads%num_query_groups == 0
    assert d_model%num_query_heads == 0

    d_head = d_model//num_query_heads
    batch_size, seq_len, d_model = q.shape

    W_q = nn.Linear(d_model, d_model, bias=False).to(device=q.device)
    W_k = nn.Linear(d_model, num_query_groups*d_head, bias = False).to(device=q.device)
    W_v = nn.Linear(d_model, num_query_groups*d_head, bias = False).to(device=q.device)
    W_out = nn.Linear(d_model, d_model, bias=True).to(device=q.device)


    q = W_q(q).view(batch_size, seq_len, num_query_heads, d_head).transpose(1, 2)
    k = W_k(k).view(batch_size, seq_len, num_query_groups, d_head).transpose(1, 2)
    v = W_v(v).view(batch_size, seq_len, num_query_groups, d_head).transpose(1, 2)

    repeat_factor = num_query_heads // num_query_groups
    k = k.repeat_interleave(repeat_factor, dim=1)
    v = v.repeat_interleave(repeat_factor, dim=1)


    scores = torch.matmul(q, k.transpose(-2,-1))/torch.sqrt(torch.tensor(d_head, device=q.device))

    if mask is not None:
      scores = scores.masked_fill(mask==False, float('-inf'))

    logits = F.softmax(scores, dim = -1)

    attn = torch.matmul(logits, v)
    out = attn.contiguous().view(batch_size, seq_len, d_model)

    return out

In [50]:
# GQA with as many K/V groups as query heads is exactly MHA, so it should match
# torch.nn.MultiheadAttention up to the (random) projection weights - we compare
# shapes here and check the grouping behaviour separately.
num_query_heads = 4
d_model = 8

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)

for num_query_groups in (1, 2, 4):
    out = grouped_query_attention(
        q, k, v,
        num_query_heads=num_query_heads,
        num_query_groups=num_query_groups,
        d_model=d_model,
    )
    assert out.shape == (batch_size, seq_len, d_model), out.shape
    print(f"num_query_groups={num_query_groups}: {tuple(out.shape)}")

# A causal mask must stop position 0 from seeing later tokens.
causal = torch.tril(torch.ones(seq_len, seq_len)).bool()
masked = grouped_query_attention(
    q, k, v, num_query_heads=num_query_heads, num_query_groups=2,
    d_model=d_model, mask=causal,
)
print("masked output:", tuple(masked.shape))



num_query_groups=1: (3, 4, 8)
num_query_groups=2: (3, 4, 8)
num_query_groups=4: (3, 4, 8)
masked output: (3, 4, 8)


In [54]:
k.repeat_interleave(2, dim=1)

tensor([[[0.9901, 0.2621, 0.2426, 0.4739, 0.4095, 0.6588, 0.9620, 0.6712],
         [0.9901, 0.2621, 0.2426, 0.4739, 0.4095, 0.6588, 0.9620, 0.6712],
         [0.0816, 0.3292, 0.4007, 0.0447, 0.6546, 0.9365, 0.7483, 0.4901],
         [0.0816, 0.3292, 0.4007, 0.0447, 0.6546, 0.9365, 0.7483, 0.4901],
         [0.4619, 0.2659, 0.4445, 0.9385, 0.2429, 0.5800, 0.1331, 0.3633],
         [0.4619, 0.2659, 0.4445, 0.9385, 0.2429, 0.5800, 0.1331, 0.3633],
         [0.5290, 0.1225, 0.6358, 0.0706, 0.2957, 0.8479, 0.9690, 0.4609],
         [0.5290, 0.1225, 0.6358, 0.0706, 0.2957, 0.8479, 0.9690, 0.4609]],

        [[0.1426, 0.0895, 0.3771, 0.2448, 0.8975, 0.1436, 0.8010, 0.5951],
         [0.1426, 0.0895, 0.3771, 0.2448, 0.8975, 0.1436, 0.8010, 0.5951],
         [0.3479, 0.9076, 0.7782, 0.4825, 0.1611, 0.3528, 0.6612, 0.9332],
         [0.3479, 0.9076, 0.7782, 0.4825, 0.1611, 0.3528, 0.6612, 0.9332],
         [0.3313, 0.2922, 0.0115, 0.8073, 0.9880, 0.8442, 0.2214, 0.8582],
         [0.3313, 0.292

In [56]:
x=torch.rand(2,3)

In [58]:
x

tensor([[0.3879, 0.6808, 0.0864],
        [0.9108, 0.8928, 0.3600]])

In [60]:
x.repeat_interleave(2, dim=0)

tensor([[0.3879, 0.6808, 0.0864],
        [0.3879, 0.6808, 0.0864],
        [0.9108, 0.8928, 0.3600],
        [0.9108, 0.8928, 0.3600]])